In [26]:
from read_data import cleaned_df
import plotly.express as px
from plotly.subplots import make_subplots
import numpy as np
import pandas as pd

In [27]:
data_path = "..\\superstore_dataset\\cleaned_Superstore.csv"
df = pd.read_csv(data_path, parse_dates=['Order Date', 'Ship Date'])

In [28]:
# Profit margin
df["Profit Margin (%)"] = (df["Profit"] / cleaned_df["Sales"]) * 100
df["Profit Margin (%)"] = df["Profit Margin (%)"].round(2)
df[["Sales", "Profit"]] = df[["Sales", "Profit"]].round(2)
df['Month'] = df['Order Date'].dt.month
df['Year'] = df['Order Date'].dt.year
df["Product_Key"] = df["Product ID"] + " | " + df["Product Name"]
# Original Unit Price
df["Original Unit Price"] = df["Sales"] / ((1 - df["Discount"]) * df["Quantity"])
df['Month_Name'] = pd.to_datetime(df['Month'], format='%m').dt.strftime('%b')

In [29]:
df_2017 = df[df["Year"] == 2017]

sales_2017 = df_2017["Sales"].sum()
print(f"Total sales in 2017: ${sales_2017:,.2f}")
profit_2017 = df_2017["Profit"].sum()
print(f"Total profit in 2017: ${profit_2017:,.2f}")
quantity_2017 = df_2017["Quantity"].sum()
print(f"Total profit in 2017: ${quantity_2017:,.2f}")


Total sales in 2017: $733,215.05
Total profit in 2017: $93,439.34
Total profit in 2017: $12,476.00


In [30]:
df.columns

Index(['Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer Name',
       'Segment', 'City', 'State', 'Postal Code', 'Region', 'Product ID',
       'Category', 'Sub-Category', 'Product Name', 'Sales', 'Quantity',
       'Discount', 'Profit', 'Ship_Duration', 'Profit Margin (%)', 'Month',
       'Year', 'Product_Key', 'Original Unit Price', 'Month_Name'],
      dtype='object')

In [31]:
print(df_2017.dtypes)

Order ID                       object
Order Date             datetime64[ns]
Ship Date              datetime64[ns]
Ship Mode                      object
Customer Name                  object
Segment                        object
City                           object
State                          object
Postal Code                     int64
Region                         object
Product ID                     object
Category                       object
Sub-Category                   object
Product Name                   object
Sales                         float64
Quantity                        int64
Discount                      float64
Profit                        float64
Ship_Duration                   int64
Profit Margin (%)             float64
Month                           int32
Year                            int32
Product_Key                    object
Original Unit Price           float64
Month_Name                     object
dtype: object


In [32]:
# Group by Category
grouped = (
    df_2017.groupby("Category", as_index=False)
    .agg({
        "Sales": "sum",
        "Profit": "sum",
        "Quantity": "sum"
    })
)

In [33]:
# ✅ Ensure month order: Jan → Dec
month_order = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
               "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

df_2017["Month_Name"] = pd.Categorical(
    df_2017["Month_Name"],
    categories=month_order,
    ordered=True
)

# ✅ Aggregate profit by Category × Month
heat_data = (
    df_2017.groupby(["Category", "Month_Name"], as_index=False)["Profit"]
    .sum()
)

# Pivot to heatmap matrix
heat_matrix = heat_data.pivot(index="Category", columns="Month_Name", values="Profit")

# ✅ Plot Heatmap
fig = px.imshow(
    heat_matrix,
    text_auto=True,
    aspect="auto",
    color_continuous_scale="Blues",
    labels=dict(color="Total Profit ($)"),
    title="Monthly Profit made by Categories (2017)",
)

fig.update_layout(
    # width=950,
    # height=350,
    xaxis_title="Month",
    yaxis_title="Category",
    margin=dict(l=60, r=40, t=60, b=60),
    coloraxis_colorbar=dict(title="Profit ($)")
)
fig.show()


C:\Users\ntxuy\AppData\Local\Temp\ipykernel_20624\2273962088.py:5: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

C:\Users\ntxuy\AppData\Local\Temp\ipykernel_20624\2273962088.py:13: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



In [34]:
import plotly.express as px
import plotly.graph_objects as go  # Assuming you imported go earlier for update_layout

# The Scatter plot definition is unchanged
fig = px.scatter(
    df_2017,
    x="Profit",
    y="Sales",
    color="Category",
    hover_data=[
        "Product Name",
        "Sub-Category",
        "Quantity",
        "Discount",
        "Month_Name"
    ],
    labels={
        "Sales": "Sales ($)",
        "Profit": "Profit ($)",
        "Category": "Product Category"
    },
    title="Sales vs Profit by Product Category (2017)",
)

# Styling markers (unchanged)
fig.update_traces(
    marker=dict(size=9, line=dict(width=1, color="white"), opacity=0.8)
)

# Layout and readability (unchanged)
fig.update_layout(
    width=950,
    height=350,
    plot_bgcolor="white",
    legend=dict(
        title="Category",
        orientation="h",
        y=1.08,
        x=0.5,
        xanchor="center",
        font=dict(size=13)
    ),
    margin=dict(l=60, r=40, t=60, b=60),
)

# --- Gridlines, Axis Lines, and ZEROLINES (MODIFIED) ---

# X-Axis (Vertical Line at x=0)
fig.update_xaxes(
    title="Sales ($)",
    showgrid=False,
    gridcolor="rgba(0,0,0,0.12)",
    griddash="dash",
    zeroline=True,
    zerolinecolor='black',  # Choose a distinct color
    zerolinewidth=1,  # Make it thicker than the border line
)

# Y-Axis (Horizontal Line at y=0)
fig.update_yaxes(
    title="Profit ($)",
    showgrid=True,
    gridcolor="rgba(0,0,0,0.12)",
    griddash="dash",
    zeroline=True,
    zerolinecolor='black',  # Choose a distinct color
    zerolinewidth=1,  # Make it thicker than the border line
)

fig.show()

In [35]:

# ✅ 1) One source of truth for category order + colors
CAT_ORDER = ["Furniture", "Office Supplies", "Technology"]
CAT_COLORS = {
    "Furniture": "#007bff",  # same as your mapping
    "Office Supplies": "#ffa600",
    "Technology": "#2ca02c"
}
fig = px.scatter(
    grouped,
    x="Sales",
    y="Profit",
    size="Quantity",
    color="Category",
    hover_name="Category",
    text="Category",
    size_max=60,
    title=None,  # we'll set a styled title below
    labels={"Sales": "Total Sales", "Profit": "Total Profit", "Quantity": "Total Quantity"},
    category_orders={"Category": CAT_ORDER},
    color_discrete_map=CAT_COLORS
)

fig.update_traces(
    textposition="middle center",
    textfont=dict(size=12, color="black"),
    opacity=0.85
)

sales_ticks = sorted(grouped["Sales"].round(0).unique())
profit_ticks = sorted(grouped["Profit"].round(0).unique())

# Put a bit more room on top; also anchor title safely inside the plotting area
fig.update_layout(
    title=dict(
        text="Sales, Profit & Quantity Distribution by Category — 2017",
        x=0.5, xanchor="center",
        y=0.97, yanchor="top",  # lower the title slightly
        font=dict(size=16),
        pad=dict(t=6, b=0, l=0, r=0)  # extra breathing room
    ),
    showlegend=False,
    xaxis=dict(
        title="Sales ($)",
        showgrid=True,
        tickvals=sales_ticks,
        gridcolor="lightgrey",
        gridwidth=0.5,
        griddash="dot"
    ),
    yaxis=dict(
        title="Profit($)",
        showgrid=True,
        tickvals=profit_ticks,
        gridcolor="lightgrey",
        gridwidth=0.5,
        griddash="dot"
    ),
    plot_bgcolor="white",
    width=500,
    height=350,
    margin=dict(l=60, r=40, t=80, b=50)  # ↑ increase top margin to avoid clipping
)

# Let axes auto-adjust margins if labels get tight
fig.update_xaxes(automargin=True)
fig.update_yaxes(automargin=True)

fig.show()

In [36]:

# Define colors for clarity
SALES_COLOR = "rgba(110, 150, 180, 0.8)"  # Muted Blue/Teal
PROFIT_COLOR = "#FF9966"  # Soft Coral/Orange

# ✅ Aggregate by month
# Assuming 'df' is defined and accessible with 'Month', 'Sales', and 'Profit' columns.
monthly = (
    df_2017.groupby("Month", as_index=False)[["Sales", "Profit"]]
    .sum()
    .sort_values("Month")
)

# Convert the integer month (1-12) to a datetime object, then format it to 'Jan', 'Feb', etc.
monthly['Month_Name'] = pd.to_datetime(monthly['Month'], format='%m').dt.strftime('%b')

# --- Create dual-axis chart ---
fig = make_subplots(specs=[[{"secondary_y": True}]])

# 🎨 Mild Color 1 (Sales Bars)
fig.add_trace(
    go.Bar(
        x=monthly["Month_Name"],
        y=monthly["Sales"],
        name="Total Sales",
        marker_color=SALES_COLOR,
        hovertemplate="Month: %{x}<br>Sales: $%{y:,.0f}<extra></extra>",
        width=0.3
    ),
    secondary_y=False
)

# 🎨 Mild Color 2 (Profit Line)
fig.add_trace(
    go.Scatter(
        x=monthly["Month_Name"],
        y=monthly["Profit"],
        name="Total Profit",
        mode="lines+markers",
        line=dict(color=PROFIT_COLOR, width=2.5),
        hovertemplate="Month: %{x}<br>Profit: $%{y:,.0f}<extra></extra>",
    ),
    secondary_y=True
)

# --- Layout ---
fig.update_layout(
    title="Monthly Sales and Profit in 2017",
    barmode="group",
    bargap=0.3,
    plot_bgcolor="white",
    legend=dict(
        orientation="h",
        y=1.1,
        x=0.05,
        bgcolor="rgba(255, 255, 255, 0.5)",
        bordercolor="lightgrey",
        borderwidth=1
    ),
    margin=dict(l=60, r=60, t=80, b=50),
)

# --- Axes ---
fig.update_xaxes(title_text="Month", showgrid=False)

# 🟦 Primary Y-Axis (Sales) -> Color-matched to Bars
fig.update_yaxes(
    title_text="Sales ($)",
    tickformat="$,.0f",
    gridcolor="lightgrey",
    griddash="dash",
    secondary_y=False,
    # Apply bar color to axis title and ticks
    title_font_color=SALES_COLOR.replace('0.8', '1.0').replace('rgba', 'rgb'),  # Use solid color for font
    tickfont_color=SALES_COLOR.replace('0.8', '1.0').replace('rgba', 'rgb')
)

# 🟧 Secondary Y-Axis (Profit) -> Color-matched to Line
fig.update_yaxes(
    title_text="Profit ($)",
    tickformat="$,.0f",
    gridcolor="lightgrey",
    secondary_y=True,
    # Apply line color to axis title and ticks
    title_font_color=PROFIT_COLOR,
    tickfont_color=PROFIT_COLOR
)

fig.show()

In [37]:
grouped2 = (
    df_2017.groupby(["Product Name", "Category", "Sub-Category"], as_index=False)
    .agg({"Sales": "sum", "Profit": "sum"})
)
top10 = grouped2.sort_values("Profit", ascending=False).head(10)
profit_order = top10["Product Name"].tolist()

# ---------- Heatmap prep (Profit Margin by Discount for the same Top-10) ----------
df_top10 = df_2017[df_2017["Product Name"].isin(profit_order)].copy()
df_top10["Discount"] = df_top10["Discount"].round(2)

summary_by_discount = (
    df_top10
    .groupby(["Discount", "Product Name"], as_index=False)
    .agg(
        **{
            "Avg Profit Margin (%)": ("Profit Margin (%)", "mean"),
            "Count": ("Profit", "size"),
            "Total Quantity": ("Quantity", "sum"),
        }
    )
)

# keep Y order
y_vals = profit_order

# Pivot for heatmap Z
z_margin = (
    summary_by_discount
    .pivot_table(index="Product Name", columns="Discount",
                 values="Avg Profit Margin (%)", aggfunc="mean")
    .reindex(index=y_vals, fill_value=np.nan)
)

# Drop all-NaN discount columns (keeps only discounts present for top-10)
z_margin_filtered = z_margin.dropna(axis=1, how='all')

x_vals = z_margin_filtered.columns.tolist()
tick_text = [f"{x * 100:.0f}%" for x in x_vals]

# Heatmap text labels
text_values = z_margin_filtered.round(2).astype(str).replace("nan", "")

# ---------- Colors ----------
custom_blue_scale = [
    [0.0, "#99c9ff"],
    [0.5, "#4da6ff"],
    [1.0, "#0059b3"],
]
blue_title = "#0059b3"
orange = "orange"
orange_dark = "#b34700"

# ---------- Build subplots with shared Y ----------
fig = make_subplots(
    rows=1, cols=2,
    shared_yaxes=True,
    column_widths=[0.55, 0.45],
    horizontal_spacing=0.08,
    specs=[[{"type": "xy"}, {"type": "heatmap"}]],
    # subplot_titles=("Top 10 Most Profitable Products & Sales (2017)",
    #                 "Profit Margin (%) by Discount")
)

# ----- Left subplot: Profit bars (main axis) -----
profit_trace = go.Bar(
    x=top10["Profit"],
    y=top10["Product Name"],
    orientation="h",
    name="Profit",
    marker=dict(color=top10["Profit"], opacity=0.8, colorscale=custom_blue_scale, showscale=False),
    width=0.8,
    text=top10["Profit"].map("{:,.0f}".format),
    textposition="none",
    textfont=dict(size=13, color="#003366"),
    cliponaxis=False,
    customdata=top10[["Category", "Sub-Category", "Sales"]],
    hovertemplate=(
        "<b>%{y}</b><br>"
        "Profit: %{x:,.2f}<br>"
        "Category: %{customdata[0]}<br>"
        "Sub-Category: %{customdata[1]}<br>"
        "Sales: %{customdata[2]:,.2f}<extra></extra>"
    ),
)
fig.add_trace(profit_trace, row=1, col=1)

# ----- Left subplot overlay: Sales mini-bars on a separate top x-axis -----
sales_trace = go.Bar(
    x=top10["Sales"],
    y=top10["Product Name"],
    orientation="h",
    name="Sales",
    marker=dict(color=orange),
    width=0.2,
    text=top10["Sales"].map("{:,.0f}".format),
    textposition="outside",
    outsidetextfont=dict(size=10, color=orange_dark, family="Arial Black"),
    insidetextanchor="start",
    cliponaxis=False,
)
# We'll attach this to a custom x-axis (xaxis3) that overlays xaxis in the left subplot.
fig.add_trace(sales_trace, row=1, col=1)

# ----- Right subplot: Heatmap (shares Y with left) -----
heatmap = go.Heatmap(
    z=z_margin_filtered.values,
    x=x_vals,
    y=y_vals,
    text=text_values.values,
    texttemplate="%{text}",
    textfont=dict(size=12, color="black"),
    colorscale="RdYlGn",
    reversescale=False,
    zmid=0,
    colorbar=dict(title="Profit Margin (%)"),
    hovertemplate="Discount: %{x}<br>Product: %{y}<br>Profit Margin: %{z:.2f}%<extra></extra>"
)
fig.add_trace(heatmap, row=1, col=2)

# ---------- Axis ranges & layout ----------
x_max_profit = float(top10["Profit"].max())
x_max_sales = float(top10["Sales"].max())

# Force the left subplot to use a specific domain so we can overlay a top x-axis (xaxis3) cleanly
fig.update_layout(
    # Domains: left subplot ~ 0 to 0.55, right subplot ~ 0.55 to 1.0 (matching column_widths)
    xaxis=dict(  # Profit axis (bottom) in left subplot
        title="Total Profit ($)",
        color=blue_title,
        tickfont=dict(color=blue_title),
        showgrid=True, gridcolor="lightgrey", gridwidth=0.4,
        range=[0, x_max_profit * 1.35],
        domain=[0.0, 0.55]
    ),
    yaxis=dict(  # Shared Y controls the category order for both
        title="",
        type='category',
        categoryorder='array',
        categoryarray=y_vals,
        autorange="reversed",  # top product at top
    ),
    # Create an overlaid top x-axis for Sales (still in the left subplot's domain)
    xaxis3=dict(
        title="Total Sales ($)",
        tickfont=dict(color=orange),
        color=orange,
        overlaying="x",
        side="top",
        anchor="y",
        showgrid=False,
        range=[0, x_max_sales * 1.18],
        matches=None,  # ensure it's independent from Profit scale
        scaleanchor=None,
        constrain="range"
    ),
    # Right subplot x-axis (discounts)
    xaxis2=dict(
        title="Discount",
        tickvals=x_vals,
        ticktext=[f"{v * 100:.0f}%" for v in x_vals],
        type='category',
        showgrid=False,
        domain=[0.60, 1.0]  # small gap equals horizontal_spacing
    ),
    # Hide Y tick labels on the heatmap side (they're shared from the left)
    yaxis2=dict(showticklabels=False, showgrid=False),
    barmode="overlay",
    uniformtext=dict(mode="show", minsize=4),
    showlegend=False,
    plot_bgcolor="white",
    margin=dict(l=140, r=120, t=150, b=50),
    title_text="Top 10 Profit Products (2017): Profit & Sales (left) + Profit Margin by Discount (right)"
)

# Make sure the second (sales) trace uses the top overlay axis
fig.data[1].update(xaxis="x3", yaxis="y")

fig.show()


In [38]:
# Table content by default
result = df_2017[df_2017["Product Name"].eq("Canon imageCLASS 2200 Advanced Copier")]

# (optional) using query:
# result = top102.query("`Product Name` == 'Canon imageCLASS 2200 Advanced Copier'")
result

,Order ID,Order Date,Ship Date,Ship Mode,Customer Name,Segment,City,State,Postal Code,Region,...,Quantity,Discount,Profit,Ship_Duration,Profit Margin (%),Month,Year,Product_Key,Original Unit Price,Month_Name
2623,CA-2017-127180,2017-10-22,2017-10-24,First Class,Tom Ashbrook,Home Office,New York City,New York,10024,East,...,4,0.2,3919.99,2,35.0,10,2017,TEC-CO-10004722 | Canon imageCLASS 2200 Advanc...,3499.990625,Oct
4190,CA-2017-166709,2017-11-17,2017-11-22,Standard Class,Hunter Lopez,Consumer,Newark,Delaware,19711,East,...,3,0.0,5039.99,5,48.0,11,2017,TEC-CO-10004722 | Canon imageCLASS 2200 Advanc...,3499.990000,Nov
8153,CA-2017-140151,2017-03-23,2017-03-25,First Class,Raymond Buch,Consumer,Seattle,Washington,98115,West,...,4,0.0,6719.98,2,48.0,3,2017,TEC-CO-10004722 | Canon imageCLASS 2200 Advanc...,3499.990000,Mar


## Combine top 10 products with heatmap